In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from scipy.spatial.distance import cdist
from scipy import stats
import matplotlib.pyplot as plt
import pymannkendall as mk
import gc 


path1=' '

in_path=path1+" "
out_path=path1+" "

som_grids = {'grid3x4': (3, 4),}   #  'grid3x3': (3, 3), 'grid4x4': (4, 4),
grid_name='grid3x4'
grid_size=(3, 4)

thresholds=(0.5,0.75,0.9)

In [ ]:
threshold=thresholds[0]

node_len=grid_size[0]*grid_size[1]
print(grid_name)
weights=pd.read_csv(out_path+f"som_results/gph_{grid_name}_total_weights.csv").values

dist_matrix = cdist(weights, weights, metric='euclidean')

#matrix = pd.DataFrame(dist_matrix) 
#matrix.to_csv(in_path+f"som_results/gph_{grid_name}_total_distance.csv", index=False)
classif=pd.read_csv(out_path+f"som_results/gph_{grid_name}_total_classif.csv").values.flatten()   #(16060,)
diff_arr = np.diff(classif) #(16059,)

node_ind=np.where(diff_arr == 0)[0]

one_series=np.diff(node_ind)
one_series[one_series!=1]=0

e_start=node_ind[np.where((np.diff(one_series)== 1)&(one_series[1:]==1))[0]+1]  
e_end=node_ind[np.where(np.diff(one_series)==-1)[0]+1]+1

ind_end=np.where(node_ind==(e_start[-1]))[0]
ind_start=np.where(node_ind==(e_end[0]-1))[0]

if np.all(one_series[:ind_start[0]] == 1):
    e_start=np.insert(e_start,0, node_ind[0])
if np.all(one_series[ind_end[0]:] == 1):
    e_end=np.insert(e_end,len(e_end), node_ind[-1])

dur=e_end-e_start
lde_ind=np.where(dur>=3)[0]
lde_s=e_start[lde_ind]
lde_e=e_end[lde_ind]
print('lde_num', len(lde_e))
node_c=[]
node_n=[]
for i in lde_e:
    node_c.append(classif[i])
    node_n.append(classif[i+2])

distance=dist_matrix

eu_dis=[]
for i in range(len(node_c)):
    eu_dis.append(distance[(node_c[i]-1),(node_n[i]-1)])
quan_dis=np.quantile(np.array(eu_dis),threshold)
print('quan_dis:', quan_dis)

n_trans=np.vstack((node_c,node_n,lde_s,lde_e, eu_dis)).T   #current node, next node, lde start, lde end

trans_df = pd.DataFrame(n_trans, columns=['node_c', 'node_n', 'lde_s', 'lde_e', 'eu_dis'])
group = trans_df.groupby('node_c').agg({'node_n': list, 'lde_s': list, 'lde_e': list, 'eu_dis': list})
#print(group)


In [ ]:
#total trans bar
all_eu_dis = []
for index, row in group.iterrows():
    all_eu_dis.extend(row['eu_dis'])
print('lde_num',len(all_eu_dis))
quan_dis=np.quantile(np.array(all_eu_dis),threshold)
print('quan_dis:',quan_dis)

fig2, axs = plt.subplots(grid_size[0], grid_size[1],figsize=(22, 12))
for i in range(grid_size[0]):
    for j in range(grid_size[1]):
        
        node=i*grid_size[1]+j+1
        print(node)

        if node not in group.index:
            node_n_ = []  
        else:
            node_n_ = group.loc[node, 'node_n']
        
        ax=axs[i,j]
        x=np.arange(1,node_len+1)
        y=[]

        if node_n_:  # 即 node_n_ 不是空的
            for n in range(node_len):
                num = np.sum(np.array(node_n_) == (n + 1))
                y.append(num)

            bars = ax.bar(x, y, width=1, edgecolor="black", linewidth=0.5)
            #print('num',y)

            for n in range(node_len):
                if distance[node - 1, n] > quan_dis:
                    bars[n].set_fc('C1')
        
        ax.set_xlabel('Node number',fontsize=14)
        ax.set_ylabel('Number of samples',fontsize=14)

        ax.set_xticks((np.arange(1,node_len+1,1)))
        ax.set_xlim([0.5,node_len+0.5])
        ax.set_yticks(np.arange(0,31,10))
        ax.tick_params(axis='both', direction='in',length=3,labelsize=14,pad=6)

        ax.annotate(f'#{node}', xy=(0.96, 0.88),xycoords='axes fraction',horizontalalignment='right',fontsize=19,color='black')

plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)

output_dir = out_path+' '

fig2.savefig(output_dir+"gph_"+grid_name+"_total_wwe.png",dpi=1000,bbox_inches='tight')
plt.show()